# Task 2 Evaluator / Viva Notebook ? Yelp Polarity

**Member:** Anshika Goel  
**Team:** 15

This notebook evaluates the **frozen final Task 2 evidence** without
rerunning training or the original final-evaluation notebooks.

It will:

- verify all three committed checkpoint SHA256 hashes;
- verify and load the frozen 38,000-row
  `final_test_predictions.csv`;
- recompute classification and calibration metrics directly from
  frozen probabilities/predictions;
- recompute confusion matrices and robustness-slice macro-F1/error rate;
- independently recompute the frozen 2,000-replicate bootstrap
  confidence intervals using the committed protocol;
- independently recompute the frozen baseline-vs-experimental
  McNemar tests and Holm adjustment;
- display the frozen 20-case manual error-review evidence;
- display historical training resource measurements.

**Scientific boundary:** the official test set has already been
consumed. This notebook does not reload raw official-test data,
does not rerun Notebook 05 or Notebook 06, and does not change
thresholds, slices, hyperparameters, or checkpoints.


In [ ]:
from __future__ import annotations

import hashlib
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import chi2
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    brier_score_loss,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)


def find_repo_root():
    current = Path.cwd().resolve()

    for candidate in [
        current,
        *current.parents,
    ]:
        expected = (
            candidate
            / "task2_sentiment"
            / "Anshika_Goel"
            / "configs"
            / "sentiment.yaml"
        )

        if expected.is_file():
            return candidate

    raise RuntimeError(
        "Unable to locate repository root."
    )


def sha256_file(path):
    digest = hashlib.sha256()

    with Path(path).open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(
                1024 * 1024
            ),
            b"",
        ):
            digest.update(chunk)

    return digest.hexdigest()


ROOT = find_repo_root()

MEMBER = (
    ROOT
    / "task2_sentiment"
    / "Anshika_Goel"
)

PREDICTIONS_PATH = (
    MEMBER
    / "outputs"
    / "predictions"
    / "final_test_predictions.csv"
)

FINAL_METRICS_PATH = (
    MEMBER
    / "outputs"
    / "metrics"
    / "final_test_metrics.json"
)

BOOTSTRAP_PATH = (
    MEMBER
    / "outputs"
    / "metrics"
    / "bootstrap_confidence_intervals.json"
)

MCNEMAR_PATH = (
    MEMBER
    / "outputs"
    / "metrics"
    / "mcnemar_tests.json"
)

ROBUSTNESS_PATH = (
    MEMBER
    / "outputs"
    / "metrics"
    / "robustness_slice_metrics.json"
)

MANUAL_SELECTION_PATH = (
    MEMBER
    / "outputs"
    / "manual_error_analysis"
    / "manual_review_selection_manifest.json"
)

MANUAL_TEXT_PATH = (
    MEMBER
    / "outputs"
    / "manual_error_analysis"
    / "manual_review_text_snapshot.json"
)

FAILURE_ANALYSIS_PATH = (
    MEMBER
    / "failure_analysis.md"
)

CHECKPOINT_PATHS = {
    "baseline_mean_pool": (
        MEMBER
        / "checkpoints"
        / "baseline_mean_pool_best.pt"
    ),
    "textcnn": (
        MEMBER
        / "checkpoints"
        / "textcnn_best.pt"
    ),
    "bigru_attention": (
        MEMBER
        / "checkpoints"
        / "bigru_attention_best.pt"
    ),
}

MODEL_ORDER = [
    "baseline_mean_pool",
    "textcnn",
    "bigru_attention",
]

EXPECTED_PREDICTION_SHA256 = (
    "9d912cfe0e743d1468d3b41ba64ed93e"
    "9f54b5e556021a1c0d87d05d11072bc4"
)

print("Repository located.")
print(
    "Task 2 member folder:",
    MEMBER.relative_to(ROOT),
)


In [ ]:
required_paths = [
    PREDICTIONS_PATH,
    FINAL_METRICS_PATH,
    BOOTSTRAP_PATH,
    MCNEMAR_PATH,
    ROBUSTNESS_PATH,
    MANUAL_SELECTION_PATH,
    MANUAL_TEXT_PATH,
    FAILURE_ANALYSIS_PATH,
    *CHECKPOINT_PATHS.values(),
]

for path in required_paths:
    if not path.exists():
        raise FileNotFoundError(
            f"Required artifact missing: "
            f"{path.relative_to(ROOT)}"
        )


final_metrics = json.loads(
    FINAL_METRICS_PATH.read_text(
        encoding="utf-8"
    )
)

bootstrap_artifact = json.loads(
    BOOTSTRAP_PATH.read_text(
        encoding="utf-8"
    )
)

mcnemar_artifact = json.loads(
    MCNEMAR_PATH.read_text(
        encoding="utf-8"
    )
)

robustness_artifact = json.loads(
    ROBUSTNESS_PATH.read_text(
        encoding="utf-8"
    )
)

manual_selection = json.loads(
    MANUAL_SELECTION_PATH.read_text(
        encoding="utf-8"
    )
)

manual_text = json.loads(
    MANUAL_TEXT_PATH.read_text(
        encoding="utf-8"
    )
)


assert (
    sha256_file(
        PREDICTIONS_PATH
    )
    == EXPECTED_PREDICTION_SHA256
)


checkpoint_rows = []

for model_name in MODEL_ORDER:

    path = CHECKPOINT_PATHS[
        model_name
    ]

    actual_sha = sha256_file(
        path
    )

    expected_sha = str(
        final_metrics[
            "inference_contract"
        ][
            model_name
        ][
            "checkpoint_sha256"
        ]
    ).lower()

    matches = (
        actual_sha
        == expected_sha
    )

    checkpoint_rows.append(
        {
            "Model": model_name,
            "Checkpoint": str(
                path.relative_to(ROOT)
            ),
            "SHA256 verified":
                matches,
        }
    )

    if not matches:
        raise RuntimeError(
            "Checkpoint SHA mismatch: "
            + model_name
        )


display(
    pd.DataFrame(
        checkpoint_rows
    )
)

print()
print(
    "CHECKPOINT / PREDICTION PROVENANCE: PASSED"
)


In [ ]:
predictions = pd.read_csv(
    PREDICTIONS_PATH
)

expected_columns = [
    "test_row",
    "source_index",
    "true_label",
    "effective_length",
    "oov_count",
    "oov_rate",
    "slice_short_reviews",
    "slice_medium_reviews",
    "slice_long_reviews",
    "slice_contains_negation",
    "slice_high_oov_rate",
    "baseline_mean_pool_probability",
    "baseline_mean_pool_prediction",
    "baseline_mean_pool_correct",
    "textcnn_probability",
    "textcnn_prediction",
    "textcnn_correct",
    "bigru_attention_probability",
    "bigru_attention_prediction",
    "bigru_attention_correct",
]

assert (
    list(predictions.columns)
    == expected_columns
)

assert len(predictions) == 38000

assert (
    predictions[
        "source_index"
    ].nunique()
    == 38000
)

label_counts = (
    predictions[
        "true_label"
    ]
    .value_counts()
    .sort_index()
    .to_dict()
)

assert label_counts == {
    0: 19000,
    1: 19000,
}


print(
    "Rows:",
    len(predictions),
)

print(
    "Columns:",
    len(predictions.columns),
)

print(
    "Class distribution:",
    label_counts,
)

print(
    "Decision threshold:",
    final_metrics[
        "evaluation"
    ][
        "decision_threshold"
    ],
)

display(
    predictions.head(5)
)

print()
print(
    "FROZEN PREDICTION CONTRACT: PASSED"
)


In [ ]:
def expected_calibration_error(
    y_true,
    probabilities,
    bins=15,
):
    y_true = np.asarray(
        y_true,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    bin_edges = np.linspace(
        0.0,
        1.0,
        bins + 1,
    )

    predicted_labels = (
        probabilities >= 0.5
    ).astype(
        np.int64
    )

    confidences = np.maximum(
        probabilities,
        1.0 - probabilities,
    )

    correctness = (
        predicted_labels
        == y_true
    ).astype(
        np.float64
    )

    ece = 0.0

    for bin_index in range(
        bins
    ):
        lower = bin_edges[
            bin_index
        ]

        upper = bin_edges[
            bin_index + 1
        ]

        if bin_index == bins - 1:
            mask = (
                (confidences >= lower)
                & (confidences <= upper)
            )
        else:
            mask = (
                (confidences >= lower)
                & (confidences < upper)
            )

        if not np.any(mask):
            continue

        bin_fraction = float(
            np.mean(mask)
        )

        bin_accuracy = float(
            np.mean(
                correctness[mask]
            )
        )

        bin_confidence = float(
            np.mean(
                confidences[mask]
            )
        )

        ece += (
            bin_fraction
            * abs(
                bin_accuracy
                - bin_confidence
            )
        )

    return float(ece)


def recompute_classification_metrics(
    labels,
    probabilities,
    threshold=0.5,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    probabilities = np.asarray(
        probabilities,
        dtype=np.float64,
    )

    predicted = (
        probabilities >= threshold
    ).astype(
        np.int64
    )

    result = {
        "accuracy": float(
            accuracy_score(
                labels,
                predicted,
            )
        ),
        "mcc": float(
            matthews_corrcoef(
                labels,
                predicted,
            )
        ),
        "roc_auc": float(
            roc_auc_score(
                labels,
                probabilities,
            )
        ),
        "pr_auc": float(
            average_precision_score(
                labels,
                probabilities,
            )
        ),
        "brier_score": float(
            brier_score_loss(
                labels,
                probabilities,
            )
        ),
        "ece_15_bins": float(
            expected_calibration_error(
                labels,
                probabilities,
                bins=15,
            )
        ),
        "confusion_matrix": (
            confusion_matrix(
                labels,
                predicted,
                labels=[0, 1],
            )
            .astype(int)
            .tolist()
        ),
    }

    for average in [
        "macro",
        "micro",
        "weighted",
    ]:
        (
            precision,
            recall,
            f1,
            _,
        ) = (
            precision_recall_fscore_support(
                labels,
                predicted,
                average=average,
                zero_division=0,
            )
        )

        result[
            f"precision_{average}"
        ] = float(precision)

        result[
            f"recall_{average}"
        ] = float(recall)

        result[
            f"f1_{average}"
        ] = float(f1)

    return result


labels = (
    predictions[
        "true_label"
    ]
    .to_numpy(
        dtype=np.int64
    )
)

threshold = float(
    final_metrics[
        "evaluation"
    ][
        "decision_threshold"
    ]
)

recomputed_metrics = {}

comparison_rows = []


scalar_metric_names = [
    "accuracy",
    "precision_macro",
    "precision_micro",
    "precision_weighted",
    "recall_macro",
    "recall_micro",
    "recall_weighted",
    "f1_macro",
    "f1_micro",
    "f1_weighted",
    "roc_auc",
    "pr_auc",
    "mcc",
    "brier_score",
    "ece_15_bins",
]


for model_name in MODEL_ORDER:

    probability_column = (
        model_name
        + "_probability"
    )

    prediction_column = (
        model_name
        + "_prediction"
    )

    probabilities = (
        predictions[
            probability_column
        ]
        .to_numpy(
            dtype=np.float64
        )
    )

    frozen_predictions = (
        predictions[
            prediction_column
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    threshold_predictions = (
        probabilities
        >= threshold
    ).astype(
        np.int64
    )

    if not np.array_equal(
        threshold_predictions,
        frozen_predictions,
    ):
        raise RuntimeError(
            "Frozen predictions disagree "
            "with thresholded probabilities: "
            + model_name
        )


    current = (
        recompute_classification_metrics(
            labels,
            probabilities,
            threshold=threshold,
        )
    )

    recomputed_metrics[
        model_name
    ] = current


    committed = (
        final_metrics[
            "models"
        ][
            model_name
        ]
    )


    for metric_name in (
        scalar_metric_names
    ):
        difference = abs(
            float(
                current[
                    metric_name
                ]
            )
            - float(
                committed[
                    metric_name
                ]
            )
        )

        comparison_rows.append(
            {
                "Model": model_name,
                "Metric":
                    metric_name,
                "Recomputed":
                    current[
                        metric_name
                    ],
                "Committed":
                    committed[
                        metric_name
                    ],
                "Abs difference":
                    difference,
            }
        )

        if difference > 1e-10:
            raise RuntimeError(
                "Final metric mismatch: "
                f"{model_name}/"
                f"{metric_name}"
            )


    if (
        current[
            "confusion_matrix"
        ]
        != committed[
            "confusion_matrix"
        ]
    ):
        raise RuntimeError(
            "Confusion-matrix mismatch: "
            + model_name
        )


display(
    pd.DataFrame(
        comparison_rows
    )
)

print()
print(
    "TASK 2 FINAL METRIC RECOMPUTATION: PASSED"
)


In [ ]:
slice_columns = {
    "short_reviews":
        "slice_short_reviews",

    "medium_reviews":
        "slice_medium_reviews",

    "long_reviews":
        "slice_long_reviews",

    "contains_negation":
        "slice_contains_negation",

    "high_oov_rate":
        "slice_high_oov_rate",
}


robustness_rows = []


for model_name in MODEL_ORDER:

    model_predictions = (
        predictions[
            model_name
            + "_prediction"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )


    for (
        slice_name,
        column_name,
    ) in slice_columns.items():

        mask = (
            predictions[
                column_name
            ]
            .astype(bool)
            .to_numpy()
        )

        slice_labels = (
            labels[mask]
        )

        slice_predictions = (
            model_predictions[
                mask
            ]
        )

        count = int(
            mask.sum()
        )

        macro_f1 = float(
            f1_score(
                slice_labels,
                slice_predictions,
                average="macro",
                zero_division=0,
            )
        )

        error_rate = float(
            1.0
            - accuracy_score(
                slice_labels,
                slice_predictions,
            )
        )


        committed = (
            robustness_artifact[
                "models"
            ][
                model_name
            ][
                slice_name
            ]
        )


        if (
            count
            != int(
                committed[
                    "count"
                ]
            )
        ):
            raise RuntimeError(
                "Slice-count mismatch: "
                f"{model_name}/"
                f"{slice_name}"
            )

        if abs(
            macro_f1
            - float(
                committed[
                    "macro_f1"
                ]
            )
        ) > 1e-12:
            raise RuntimeError(
                "Slice macro-F1 mismatch: "
                f"{model_name}/"
                f"{slice_name}"
            )

        if abs(
            error_rate
            - float(
                committed[
                    "error_rate"
                ]
            )
        ) > 1e-12:
            raise RuntimeError(
                "Slice error-rate mismatch: "
                f"{model_name}/"
                f"{slice_name}"
            )


        robustness_rows.append(
            {
                "Model": model_name,
                "Slice": slice_name,
                "Count": count,
                "Macro-F1": macro_f1,
                "Error rate":
                    error_rate,
            }
        )


display(
    pd.DataFrame(
        robustness_rows
    )
)

print()
print(
    "ROBUSTNESS-SLICE RECOMPUTATION: PASSED"
)


In [ ]:
def metrics_from_binary_counts(
    tn,
    fp,
    fn,
    tp,
):
    total = (
        tn
        + fp
        + fn
        + tp
    )

    accuracy = (
        (tn + tp)
        / total
    )

    positive_denominator = (
        2 * tp
        + fp
        + fn
    )

    negative_denominator = (
        2 * tn
        + fp
        + fn
    )

    positive_f1 = (
        0.0
        if positive_denominator == 0
        else (
            2 * tp
            / positive_denominator
        )
    )

    negative_f1 = (
        0.0
        if negative_denominator == 0
        else (
            2 * tn
            / negative_denominator
        )
    )

    macro_f1 = (
        positive_f1
        + negative_f1
    ) / 2.0

    denominator = math.sqrt(
        (tp + fp)
        * (tp + fn)
        * (tn + fp)
        * (tn + fn)
    )

    mcc = (
        0.0
        if denominator == 0
        else (
            (
                tp * tn
                - fp * fn
            )
            / denominator
        )
    )

    return (
        float(accuracy),
        float(macro_f1),
        float(mcc),
    )


def bootstrap_confidence_intervals(
    labels,
    predictions_by_model,
    replicates,
    confidence_level,
    seed,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    model_names = list(
        predictions_by_model.keys()
    )

    prediction_matrix = np.stack(
        [
            np.asarray(
                predictions_by_model[
                    model_name
                ],
                dtype=np.int64,
            )
            for model_name
            in model_names
        ],
        axis=0,
    )

    sample_count = int(
        labels.shape[0]
    )

    generator = np.random.default_rng(
        seed
    )

    values = {
        model_name: {
            "accuracy":
                np.empty(
                    replicates,
                    dtype=np.float64,
                ),
            "macro_f1":
                np.empty(
                    replicates,
                    dtype=np.float64,
                ),
            "mcc":
                np.empty(
                    replicates,
                    dtype=np.float64,
                ),
        }
        for model_name
        in model_names
    }


    for replicate_index in range(
        replicates
    ):
        indices = generator.integers(
            low=0,
            high=sample_count,
            size=sample_count,
            dtype=np.int64,
        )

        sampled_labels = (
            labels[indices]
        )


        for (
            model_index,
            model_name,
        ) in enumerate(
            model_names
        ):
            sampled_predictions = (
                prediction_matrix[
                    model_index,
                    indices,
                ]
            )

            encoded = (
                sampled_labels * 2
                + sampled_predictions
            )

            counts = np.bincount(
                encoded,
                minlength=4,
            )

            (
                accuracy,
                macro_f1,
                mcc,
            ) = (
                metrics_from_binary_counts(
                    tn=int(counts[0]),
                    fp=int(counts[1]),
                    fn=int(counts[2]),
                    tp=int(counts[3]),
                )
            )

            values[
                model_name
            ][
                "accuracy"
            ][
                replicate_index
            ] = accuracy

            values[
                model_name
            ][
                "macro_f1"
            ][
                replicate_index
            ] = macro_f1

            values[
                model_name
            ][
                "mcc"
            ][
                replicate_index
            ] = mcc


    alpha = (
        1.0
        - confidence_level
    )

    lower_q = (
        alpha / 2.0
    )

    upper_q = (
        1.0
        - alpha / 2.0
    )

    result = {}


    for model_name in model_names:

        result[
            model_name
        ] = {}

        for (
            metric_name,
            samples,
        ) in values[
            model_name
        ].items():

            lower, upper = (
                np.quantile(
                    samples,
                    [
                        lower_q,
                        upper_q,
                    ],
                    method="linear",
                )
            )

            result[
                model_name
            ][
                metric_name
            ] = {
                "lower":
                    float(lower),
                "upper":
                    float(upper),
            }


    return result


predictions_by_model = {
    model_name: (
        predictions[
            model_name
            + "_prediction"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )
    for model_name
    in MODEL_ORDER
}


bootstrap_recomputed = (
    bootstrap_confidence_intervals(
        labels=labels,
        predictions_by_model=(
            predictions_by_model
        ),
        replicates=int(
            bootstrap_artifact[
                "replicates"
            ]
        ),
        confidence_level=float(
            bootstrap_artifact[
                "confidence_level"
            ]
        ),
        seed=int(
            bootstrap_artifact[
                "seed"
            ]
        ),
    )
)


bootstrap_rows = []


for model_name in MODEL_ORDER:

    for metric_name in [
        "accuracy",
        "macro_f1",
        "mcc",
    ]:

        recomputed_interval = (
            bootstrap_recomputed[
                model_name
            ][
                metric_name
            ]
        )

        committed_interval = (
            bootstrap_artifact[
                "intervals"
            ][
                model_name
            ][
                metric_name
            ]
        )

        for bound in [
            "lower",
            "upper",
        ]:
            difference = abs(
                float(
                    recomputed_interval[
                        bound
                    ]
                )
                - float(
                    committed_interval[
                        bound
                    ]
                )
            )

            if difference > 1e-12:
                raise RuntimeError(
                    "Bootstrap mismatch: "
                    f"{model_name}/"
                    f"{metric_name}/"
                    f"{bound}"
                )


        bootstrap_rows.append(
            {
                "Model":
                    model_name,
                "Metric":
                    metric_name,
                "Recomputed lower":
                    recomputed_interval[
                        "lower"
                    ],
                "Recomputed upper":
                    recomputed_interval[
                        "upper"
                    ],
                "Committed lower":
                    committed_interval[
                        "lower"
                    ],
                "Committed upper":
                    committed_interval[
                        "upper"
                    ],
            }
        )


display(
    pd.DataFrame(
        bootstrap_rows
    )
)

print()
print(
    "BOOTSTRAP CI RECOMPUTATION: PASSED"
)


In [ ]:
def mcnemar_continuity_corrected(
    labels,
    predictions_a,
    predictions_b,
):
    labels = np.asarray(
        labels,
        dtype=np.int64,
    )

    predictions_a = np.asarray(
        predictions_a,
        dtype=np.int64,
    )

    predictions_b = np.asarray(
        predictions_b,
        dtype=np.int64,
    )

    correct_a = (
        predictions_a
        == labels
    )

    correct_b = (
        predictions_b
        == labels
    )

    a_correct_b_wrong = int(
        np.sum(
            correct_a
            & ~correct_b
        )
    )

    a_wrong_b_correct = int(
        np.sum(
            ~correct_a
            & correct_b
        )
    )

    discordant = (
        a_correct_b_wrong
        + a_wrong_b_correct
    )

    if discordant == 0:
        statistic = 0.0
        p_value = 1.0

    else:
        statistic = (
            (
                abs(
                    a_correct_b_wrong
                    - a_wrong_b_correct
                )
                - 1.0
            )
            ** 2
            / discordant
        )

        p_value = float(
            chi2.sf(
                statistic,
                df=1,
            )
        )

    return {
        "a_correct_b_wrong":
            a_correct_b_wrong,

        "a_wrong_b_correct":
            a_wrong_b_correct,

        "discordant_pairs":
            int(discordant),

        "chi_square_statistic":
            float(statistic),

        "raw_p_value":
            float(p_value),

        "continuity_correction":
            True,
    }


def holm_adjust(
    raw_p_values,
):
    names = list(
        raw_p_values.keys()
    )

    p_values = np.asarray(
        [
            raw_p_values[name]
            for name
            in names
        ],
        dtype=np.float64,
    )

    order = np.argsort(
        p_values
    )

    adjusted = np.empty_like(
        p_values
    )

    previous = 0.0

    family_size = len(
        names
    )


    for rank, index in enumerate(
        order
    ):
        candidate = (
            (
                family_size
                - rank
            )
            * p_values[index]
        )

        candidate = min(
            1.0,
            max(
                previous,
                candidate,
            ),
        )

        adjusted[
            index
        ] = candidate

        previous = candidate


    return {
        names[index]:
            float(
                adjusted[index]
            )
        for index
        in range(
            len(names)
        )
    }


comparison_models = {
    "baseline_mean_pool_vs_textcnn":
        (
            "baseline_mean_pool",
            "textcnn",
        ),

    "baseline_mean_pool_vs_bigru_attention":
        (
            "baseline_mean_pool",
            "bigru_attention",
        ),
}


mcnemar_recomputed = {}

for (
    comparison_name,
    (
        model_a,
        model_b,
    ),
) in comparison_models.items():

    mcnemar_recomputed[
        comparison_name
    ] = (
        mcnemar_continuity_corrected(
            labels,
            predictions_by_model[
                model_a
            ],
            predictions_by_model[
                model_b
            ],
        )
    )


raw_p_values = {
    name: result[
        "raw_p_value"
    ]
    for name, result
    in mcnemar_recomputed.items()
}

adjusted = holm_adjust(
    raw_p_values
)


mcnemar_rows = []


for comparison_name in (
    comparison_models
):
    recomputed = (
        mcnemar_recomputed[
            comparison_name
        ]
    )

    recomputed[
        "holm_adjusted_p_value"
    ] = adjusted[
        comparison_name
    ]

    committed = (
        mcnemar_artifact[
            "comparisons"
        ][
            comparison_name
        ]
    )


    for integer_key in [
        "a_correct_b_wrong",
        "a_wrong_b_correct",
        "discordant_pairs",
    ]:
        if int(
            recomputed[
                integer_key
            ]
        ) != int(
            committed[
                integer_key
            ]
        ):
            raise RuntimeError(
                "McNemar count mismatch: "
                f"{comparison_name}/"
                f"{integer_key}"
            )


    for float_key in [
        "chi_square_statistic",
        "raw_p_value",
        "holm_adjusted_p_value",
    ]:
        if abs(
            float(
                recomputed[
                    float_key
                ]
            )
            - float(
                committed[
                    float_key
                ]
            )
        ) > 1e-12:
            raise RuntimeError(
                "McNemar numeric mismatch: "
                f"{comparison_name}/"
                f"{float_key}"
            )


    mcnemar_rows.append(
        {
            "Comparison":
                comparison_name,
            **recomputed,
        }
    )


display(
    pd.DataFrame(
        mcnemar_rows
    )
)

print()
print(
    "MCNEMAR / HOLM RECOMPUTATION: PASSED"
)

print()
print(
    "Statistical boundary: "
    "no TextCNN-vs-BiGRU-Attention "
    "McNemar test was part of the "
    "frozen family."
)


In [ ]:
resource_rows = []

for model_name in MODEL_ORDER:

    saved = (
        final_metrics[
            "training_resource_summary"
        ][
            model_name
        ]
    )

    resource_rows.append(
        {
            "Model":
                model_name,

            "Parameters":
                saved[
                    "parameter_count"
                ],

            "Training seconds":
                saved[
                    "training_seconds"
                ],

            "Mean train examples/s":
                saved[
                    "mean_train_examples_per_second"
                ],

            "Peak GPU allocated MB":
                saved[
                    "peak_gpu_allocated_mb"
                ],

            "Peak GPU reserved MB":
                saved[
                    "peak_gpu_reserved_mb"
                ],
        }
    )


display(
    Markdown(
        "## Historical training-resource evidence"
    )
)

display(
    pd.DataFrame(
        resource_rows
    )
)

print(
    "These are measurements from the "
    "original training runs and are "
    "not recreated by this evaluator."
)


In [ ]:
selection_records = (
    manual_selection[
        "records"
    ]
)

assert len(
    selection_records
) == 20

selection_df = pd.DataFrame(
    selection_records
)


def extract_text_records(payload):

    if isinstance(
        payload,
        list,
    ):
        return payload

    if isinstance(
        payload,
        dict,
    ):
        for key in [
            "records",
            "reviews",
            "items",
            "samples",
        ]:
            value = payload.get(
                key
            )

            if isinstance(
                value,
                list,
            ):
                return value

    return []


text_records = (
    extract_text_records(
        manual_text
    )
)

text_df = pd.DataFrame(
    text_records
)


display(
    Markdown(
        "## Frozen 20-case manual error review"
    )
)

display(
    selection_df
)


if not text_df.empty:
    display(
        Markdown(
            "### Frozen reviewed-text snapshot"
        )
    )

    display(
        text_df
    )


print(
    "Selection records:",
    len(selection_df),
)

print(
    "Focal model:",
    manual_selection[
        "focal_model"
    ],
)

print(
    "Decision threshold:",
    manual_selection[
        "decision_threshold"
    ],
)

print(
    "Selection performed before "
    "review-text access:",
    manual_selection[
        "selection_performed_before_review_text_access"
    ],
)


In [ ]:
display(
    Markdown(
        "## Task 2 documented failure analysis"
    )
)

display(
    Markdown(
        FAILURE_ANALYSIS_PATH.read_text(
            encoding="utf-8"
        )
    )
)


## Task 2 evaluator boundary

A successful run verifies that the frozen 38,000-row prediction
artifact reproduces the committed final classification/calibration
metrics, robustness slices, bootstrap confidence intervals, and
frozen McNemar family.

The three model checkpoints are verified by SHA256 but are not used
to regenerate official-test predictions in this notebook. This avoids
unnecessary repeated consumption of the official test workflow.

Historical training time, throughput, and peak-memory values remain
evidence from the original runs.

No new model-selection or tuning conclusion may be made from this
evaluator notebook.
